In [10]:
import os
import sys
import time
import yaml
import pandas as pd
import numpy as np
import json

with open('../../config.local.yaml', 'r') as f:
    local_config = yaml.safe_load(f)

LOCAL_PATH = local_config['LOCAL_PATH']
DATA_PATH = local_config['DATA_PATH']

sys.path.append(os.path.join(LOCAL_PATH, "src/python"))

from utils import is_casenum


In [11]:
meetings_df = pd.read_csv(os.path.join(LOCAL_PATH, "intermediate_data/cpc/meetings-manifest.csv"))
DATES = sorted(list(meetings_df['date']))

In [12]:
out_df = []
for idx, row in meetings_df.iterrows():
    date = row['date']
    year = date[0:4]
    input_filepath = os.path.join(DATA_PATH, "intermediate_data/cpc", year, date, "agenda-summary.json")
    override_filepath = os.path.join(LOCAL_PATH, "intermediate_data/cpc", year, date, "agenda-summary-override.json")
    if os.path.exists(override_filepath):
        with open(override_filepath, 'r') as f:
            j = json.load(f)
    elif os.path.exists(input_filepath):
        with open(input_filepath, 'r') as f:
            j = json.load(f)
    else:
        continue
    agenda_items = j["agenda_items"]
    for item in agenda_items:
        out_row = {
            "year": row["year"],
            "date": date
        }
        for k, v in item.items():
            out_row[k] = v
        out_df.append(out_row)

out_df = pd.DataFrame(out_df)
out_df.to_parquet(os.path.join(DATA_PATH, "intermediate_data/cpc/agenda-summaries.parquet"), index=False)